# Gated Power BI / Fabric Deployment Pipeline Widget

Promote items between deployment-pipeline stages only when your custom tests pass.

**Workflow**
1. Pick a deployment pipeline.
2. Pick the *item type* you want to deploy. The list is populated dynamically from whatever items the pipeline contains — Reports, Semantic models, Notebooks, Lakehouses, Warehouses, Data pipelines, KQL databases, Dataflows, and so on.
3. Pick a stage-to-stage gate (e.g. Dev → Test → Prod) and tick the tests you want to enforce.
4. Click **Run Tests** — each selected callable runs and results show inline.
5. If every selected test passes, **Deploy →** is enabled and clicks call `sempy_labs.deploy_stage_content` for the selected item.

Tests are Python callables passed explicitly via `tests={...}` to `deployment_pipeline_widget()`. Decorate them with `@dpw_test(kind="Report" | "SemanticModel" | "Notebook" | ... | "all")` to scope them to a specific Fabric item type, or to all of them.

## 1. Install dependencies

In [ ]:
%pip install -q --upgrade --force-reinstall pandas
%pip install -q --upgrade anywidget semantic-link-labs

## 2. Imports & widget assets (CSS + JS)

In [ ]:
from __future__ import annotations

import inspect
import traceback
from typing import Any, Callable, Dict, List, Optional
from uuid import UUID

import anywidget
import traitlets

In [ ]:
_WIDGET_CSS = r"""
.dpw-root {
    --bg-solid:       #ffffff;
    --surface:        rgba(255, 255, 255, 0.85);
    --surface-2:      rgba(0, 0, 0, 0.03);
    --border:         rgba(0, 0, 0, 0.08);
    --border-strong:  rgba(0, 0, 0, 0.14);
    --text:           #1d1d1f;
    --text-secondary: #6e6e73;
    --accent:         #007AFF;
    --accent-hover:   #0a6cdb;
    --accent-soft:    rgba(0, 122, 255, 0.12);
    --orange:         #FF9500;
    --success:        #34c759;
    --success-soft:   rgba(52, 199, 89, 0.14);
    --danger:         #ff3b30;
    --danger-soft:    rgba(255, 59, 48, 0.14);
    --radius:         14px;
    --radius-sm:      8px;

    font-family: -apple-system, BlinkMacSystemFont, "SF Pro Text",
                 "Helvetica Neue", Arial, sans-serif;
    font-size: 13px;
    color: var(--text);
    background: var(--bg-solid);
    border-radius: var(--radius);
    padding: 20px;
    max-width: 1100px;
    margin: 0 auto;
    box-sizing: border-box;
}
@media (prefers-color-scheme: dark) {
    .dpw-root.dpw-auto {
        --bg-solid:       #1c1c1e;
        --surface:        rgba(44, 44, 46, 0.85);
        --surface-2:      rgba(255, 255, 255, 0.04);
        --border:         rgba(255, 255, 255, 0.08);
        --border-strong:  rgba(255, 255, 255, 0.14);
        --text:           #f5f5f7;
        --text-secondary: #98989d;
    }
}
.dpw-root.dpw-dark {
    --bg-solid:       #1c1c1e;
    --surface:        rgba(44, 44, 46, 0.85);
    --surface-2:      rgba(255, 255, 255, 0.04);
    --border:         rgba(255, 255, 255, 0.08);
    --border-strong:  rgba(255, 255, 255, 0.14);
    --text:           #f5f5f7;
    --text-secondary: #98989d;
}

.dpw-root.dpw-busy { pointer-events: none; opacity: 0.55; }
.dpw-root { position: relative; }
.dpw-spinner-overlay {
    position: absolute; inset: 0;
    display: none;
    align-items: center; justify-content: center;
    flex-direction: column; gap: 10px;
    background: rgba(0,0,0,0.05);
    z-index: 50;
    pointer-events: auto;
    border-radius: var(--radius-sm);
}
.dpw-root.dpw-busy .dpw-spinner-overlay { display: flex; }
.dpw-spinner {
    width: 36px; height: 36px;
    border: 3px solid var(--border-strong);
    border-top-color: var(--accent);
    border-radius: 50%;
    animation: dpw-spin 0.8s linear infinite;
}
.dpw-spinner-label { font-size: 12px; color: var(--text-secondary); font-weight: 500; }
@keyframes dpw-spin { to { transform: rotate(360deg); } }

.dpw-h1 { margin: 0 0 4px; font-size: 18px; font-weight: 600; }
.dpw-sub { color: var(--text-secondary); margin: 0 0 16px; font-size: 12px; }

.dpw-pickers { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; margin-bottom: 18px; }
.dpw-picker { background: var(--surface-2); border: 1px solid var(--border);
              border-radius: var(--radius-sm); padding: 12px; }
.dpw-picker label { display: block; font-size: 11px; color: var(--text-secondary);
                    text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 6px; }
.dpw-picker select { width: 100%; padding: 6px 8px; border-radius: 6px;
                     border: 1px solid var(--border-strong); background: var(--bg-solid);
                     color: var(--text); font-size: 13px; }

.dpw-columns { display: grid; grid-template-columns: 1fr 1fr; gap: 18px; }
.dpw-column { display: flex; flex-direction: column; gap: 10px; }
.dpw-col-title { font-size: 13px; font-weight: 600; padding: 0 4px;
                 color: var(--text-secondary); text-transform: uppercase;
                 letter-spacing: 0.05em; }

.dpw-stage { background: var(--surface-2); border: 1px solid var(--border);
             border-radius: var(--radius-sm); padding: 10px 12px; }
.dpw-stage-name { font-weight: 600; }
.dpw-stage-meta { color: var(--text-secondary); font-size: 11px; margin-top: 2px; }
.dpw-stage-items { margin-top: 6px; font-size: 11px; color: var(--text-secondary); }

.dpw-gate { border: 1px dashed var(--border-strong); border-radius: var(--radius-sm);
            padding: 10px 12px; background: var(--surface); }
.dpw-gate-title { display: flex; align-items: center; gap: 8px;
                  font-size: 12px; font-weight: 600; color: var(--accent); }
.dpw-gate-arrow { font-size: 14px; }

.dpw-tests { margin: 8px 0; display: flex; flex-direction: column; gap: 4px; }
.dpw-test-row { display: flex; align-items: center; gap: 6px; font-size: 12px; }
.dpw-test-row code { background: var(--surface-2); padding: 1px 5px; border-radius: 4px;
                     font-size: 11px; }
.dpw-test-status { margin-left: auto; font-size: 11px; padding: 1px 7px;
                   border-radius: 10px; }
.dpw-test-status.pass { background: var(--success-soft); color: var(--success); }
.dpw-test-status.fail { background: var(--danger-soft);  color: var(--danger); }
.dpw-test-status.error { background: var(--danger-soft); color: var(--danger); }
.dpw-test-status.pending { background: var(--surface-2); color: var(--text-secondary); }

.dpw-test-err { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
                font-size: 11px; color: var(--danger); white-space: pre-wrap;
                margin: 4px 0 0 22px; }

.dpw-gate-actions { display: flex; gap: 6px; margin-top: 8px; }

.dpw-btn { display: inline-flex; align-items: center; gap: 4px;
           padding: 5px 12px; border-radius: 20px; border: none;
           font-size: 12px; font-weight: 500; cursor: pointer;
           transition: background 0.15s, opacity 0.15s; }
.dpw-btn-primary { background: var(--accent); color: #fff; }
.dpw-btn-primary:hover { background: var(--accent-hover); }
.dpw-btn-success { background: var(--success); color: #fff; }
.dpw-btn-ghost { background: transparent; color: var(--text-secondary);
                 border: 1px solid var(--border-strong); }
.dpw-btn-ghost:hover { background: var(--surface-2); }
.dpw-btn:disabled { opacity: 0.45; cursor: not-allowed; }

.dpw-empty { padding: 12px; color: var(--text-secondary); font-size: 12px;
             text-align: center; }

.dpw-status { display: none; padding: 8px 12px; border-radius: var(--radius-sm);
              font-size: 12px; margin-top: 14px; white-space: pre-wrap; }
.dpw-status.show { display: block; }
.dpw-status.success { background: var(--success-soft); color: var(--success); }
.dpw-status.error   { background: var(--danger-soft);  color: var(--danger);  }
.dpw-status.info    { background: var(--accent-soft);  color: var(--accent);  }
"""

In [ ]:
_WIDGET_JS = r"""
function render({ model, el }) {
    // Prevent duplicate widget roots when the renderer is re-invoked for the same output element.
    el.innerHTML = "";

    const root = document.createElement("div");
    root.className = "dpw-root";
    el.appendChild(root);

    const uiState = {
        activeKind: null,
        gateByKind: {},
    };

    function applyTheme() {
        root.classList.remove("dpw-dark", "dpw-auto");
        const dm = model.get("dark_mode");
        if (dm === true) root.classList.add("dpw-dark");
        else if (dm === null || dm === undefined) root.classList.add("dpw-auto");
    }
    applyTheme();
    model.on("change:dark_mode", applyTheme);

    function escapeHtml(s) {
        return String(s ?? "")
            .replace(/&/g, "&amp;")
            .replace(/</g, "&lt;")
            .replace(/>/g, "&gt;")
            .replace(/\"/g, "&quot;");
    }

    function setBusy(b, label) {
        root.classList.toggle("dpw-busy", b);
        if (b && label) {
            const lbl = root.querySelector('[data-role="spinner-label"]');
            if (lbl) lbl.textContent = label;
        }
    }

    function send(action) {
        const fastActions = new Set(["toggle_items", "toggle_test"]);
        if (!fastActions.has(action.action)) {
            const labels = {
                deploy: "Deploying... this can take a minute.",
                run_tests: "Running tests...",
                refresh_pipelines: "Refreshing pipelines...",
                select_pipeline: "Loading pipeline...",
            };
            setBusy(true, labels[action.action] || "Working...");
        }
        model.set("pending_action", action);
        model.set("run", (model.get("run") || 0) + 1);
        model.save_changes();
    }

    function setStatus(message, kind) {
        if (!statusEl) return;
        if (!message) {
            statusEl.classList.remove("show");
            return;
        }
        statusEl.className = `dpw-status show ${kind || "info"}`;
        statusEl.textContent = message;
    }

    const header = document.createElement("div");
    header.style.cssText = "display:flex;justify-content:space-between;align-items:flex-start;gap:10px;";
    header.innerHTML = `
        <div>
          <h2 class="dpw-h1">Deployment Pipeline Runner</h2>
          <p class="dpw-sub">Pick a pipeline, then an item type, then select an item, tests, and deploy.</p>
        </div>`;
    const refreshBtn = document.createElement("button");
    refreshBtn.className = "dpw-btn dpw-btn-ghost";
    refreshBtn.textContent = "Refresh pipelines";
    refreshBtn.onclick = () => send({ action: "refresh_pipelines" });
    header.appendChild(refreshBtn);
    root.appendChild(header);

    const card = document.createElement("div");
    card.className = "dpw-stage";
    card.style.marginTop = "10px";
    root.appendChild(card);

    const statusEl = document.createElement("div");
    statusEl.className = "dpw-status";
    root.appendChild(statusEl);

    const spinnerOverlay = document.createElement("div");
    spinnerOverlay.className = "dpw-spinner-overlay";
    spinnerOverlay.innerHTML = `
        <div class="dpw-spinner"></div>
        <div class="dpw-spinner-label" data-role="spinner-label">Working...</div>
    `;
    root.appendChild(spinnerOverlay);

    function getGateOptions(stages) {
        const out = [];
        for (let i = 0; i < Math.max(0, stages.length - 1); i++) {
            const src = stages[i];
            const tgt = stages[i + 1];
            out.push({
                key: `${src.id}__${tgt.id}`,
                label: `${src.name} -> ${tgt.name}`,
                source: src,
                target: tgt,
            });
        }
        return out;
    }

    function render() {
        const data = model.get("data") || {};
        const pipelines = data.pipelines || [];
        const selectedPipelineId = data.selected_pipeline || "";
        const stages = data.stages || [];
        const itemTypes = data.item_types || [];
        const itemsByStage = data.items || {};      // {stage_id: {item_type: [items]}}
        const gatesByKind = data.gates || {};       // {item_type: {gate_key: {...}}}
        const allTests = data.tests || {};          // {item_type: [{name, accepts_item}]}

        // Reconcile activeKind with available item types.
        if (!itemTypes.length) {
            uiState.activeKind = null;
        } else if (!itemTypes.some(t => t.value === uiState.activeKind)) {
            uiState.activeKind = itemTypes[0].value;
        }
        const kind = uiState.activeKind;
        const kindMeta = itemTypes.find(t => t.value === kind) || null;
        const kindLabel = kindMeta ? kindMeta.label : (kind || "");
        const tests = (kind && allTests[kind]) ? allTests[kind] : [];

        const gateOptions = getGateOptions(stages);
        if (kind) {
            const gatesForKind = gatesByKind[kind] || {};
            if (!uiState.gateByKind[kind] || !gatesForKind[uiState.gateByKind[kind]]) {
                uiState.gateByKind[kind] = gateOptions.length ? gateOptions[0].key : null;
            }
        }
        const activeGateKey = kind ? uiState.gateByKind[kind] : null;
        const activeGate = gateOptions.find(g => g.key === activeGateKey) || null;

        const pipelineOptions = ['<option value="">- none -</option>']
            .concat(pipelines.map(p => {
                const sel = p.id === selectedPipelineId ? " selected" : "";
                return `<option value="${escapeHtml(p.id)}"${sel}>${escapeHtml(p.name)}</option>`;
            }))
            .join("");

        const kindOptions = itemTypes.length
            ? itemTypes.map(t => {
                const sel = t.value === kind ? " selected" : "";
                const count = (typeof t.count === "number") ? ` (${t.count})` : "";
                return `<option value="${escapeHtml(t.value)}"${sel}>${escapeHtml(t.label)}${count}</option>`;
            }).join("")
            : '<option value="">- no items found -</option>';

        const gateSelectOptions = gateOptions.map(g => {
            const sel = g.key === activeGateKey ? " selected" : "";
            return `<option value="${escapeHtml(g.key)}"${sel}>${escapeHtml(g.label)}</option>`;
        }).join("");

        card.innerHTML = `
          <div class="dpw-pickers" style="margin-bottom:10px;">
            <div class="dpw-picker">
              <label>Pipeline</label>
              <select data-role="pipeline">${pipelineOptions}</select>
            </div>
            <div class="dpw-picker">
              <label>Item Type</label>
              <select data-role="kind" ${itemTypes.length ? "" : "disabled"}>${kindOptions}</select>
            </div>
          </div>
          ${activeGate ? `
          <div class="dpw-picker" style="margin-bottom:10px;">
            <label>Stage Gate</label>
            <select data-role="gate">${gateSelectOptions}</select>
          </div>` : (selectedPipelineId
              ? '<div class="dpw-empty">No stage-to-stage gates available for this pipeline.</div>'
              : '<div class="dpw-empty">Select a pipeline to begin.</div>')}
          <div data-role="items"></div>
          <div data-role="tests"></div>
          <div class="dpw-gate-actions" data-role="actions"></div>
        `;

        const kindSel = card.querySelector('select[data-role="kind"]');
        const pipelineSel = card.querySelector('select[data-role="pipeline"]');
        if (kindSel) {
            kindSel.onchange = (e) => {
                uiState.activeKind = e.target.value;
                render();
            };
        }
        if (pipelineSel) {
            pipelineSel.onchange = (e) => {
                send({ action: "select_pipeline", pipeline_id: e.target.value });
            };
        }

        if (!activeGate || !kind) {
            const n = data.notice || {};
            if (n.message) setStatus(n.message, n.kind || "info");
            return;
        }

        const kindGates = gatesByKind[kind] || {};
        const gateData = kindGates[activeGate.key] || { selected: [], selected_items: [], results: {} };
        const stageItemsByType = itemsByStage[activeGate.source.id] || {};
        const sourceItems = stageItemsByType[kind] || [];

        const gateSel = card.querySelector('select[data-role="gate"]');
        if (gateSel) {
            gateSel.onchange = (e) => {
                uiState.gateByKind[kind] = e.target.value;
                render();
            };
        }

        const selectedItemId = (gateData.selected_items || [])[0] || "";
        const selectedTests = new Set(gateData.selected || []);

        const itemsWrap = card.querySelector('div[data-role="items"]');
        itemsWrap.innerHTML = `<div style="font-size:11px;color:var(--text-secondary);margin:8px 0 4px;">${escapeHtml(kindLabel)} items to deploy</div>`;
        if (!sourceItems.length) {
            const empty = document.createElement("div");
            empty.className = "dpw-empty";
            empty.style.textAlign = "left";
            empty.textContent = `No ${kindLabel.toLowerCase()} items in the source stage.`;
            itemsWrap.appendChild(empty);
        } else {
            const sortedItems = [...sourceItems].sort((a, b) =>
                String(a.itemName || a.sourceItemId).localeCompare(
                    String(b.itemName || b.sourceItemId),
                    undefined,
                    { sensitivity: "base" }
                )
            );
            itemsWrap.innerHTML += `
              <div style="display:flex;gap:8px;align-items:center;margin:6px 0;">
                <input data-role="item-search" type="text" placeholder="Search items..."
                       style="flex:1;padding:6px 8px;border-radius:6px;border:1px solid var(--border-strong);background:var(--bg-solid);color:var(--text);font-size:12px;"/>
                <button class="dpw-btn dpw-btn-ghost" data-role="item-clear" style="padding:4px 10px;">Clear</button>
              </div>
              <select data-role="item-select" size="8"
                      style="width:100%;padding:6px 8px;border-radius:6px;border:1px solid var(--border-strong);background:var(--bg-solid);color:var(--text);font-size:12px;"></select>
            `;

            const searchInput = itemsWrap.querySelector('input[data-role="item-search"]');
            const itemSelect = itemsWrap.querySelector('select[data-role="item-select"]');
            const clearBtn = itemsWrap.querySelector('button[data-role="item-clear"]');

            function renderItemOptions(filterText) {
                const f = String(filterText || "").toLowerCase();
                const filtered = !f
                    ? sortedItems
                    : sortedItems.filter(it => String(it.itemName || it.sourceItemId).toLowerCase().includes(f));
                itemSelect.innerHTML = filtered
                    .map(it => {
                        const id = String(it.sourceItemId);
                        const name = String(it.itemName || id);
                        const sel = id === selectedItemId ? " selected" : "";
                        return `<option value="${escapeHtml(id)}"${sel}>${escapeHtml(name)}</option>`;
                    })
                    .join("");
            }

            renderItemOptions("");

            searchInput.oninput = () => {
                renderItemOptions(searchInput.value || "");
            };

            const onItemSelectChanged = () => {
                const value = itemSelect.value || "";
                gateData.selected_items = value ? [value] : [];
                const deployBtn = card.querySelector('[data-act=\"deploy\"]');
                if (deployBtn) {
                    deployBtn.disabled = !((gateData.selected_items || []).length === 1);
                }
                send({
                    action: "toggle_items",
                    kind: uiState.activeKind,
                    gate: activeGate.key,
                    selected_items: value ? [value] : [],
                });
            };
            itemSelect.onchange = onItemSelectChanged;

            clearBtn.onclick = () => {
                gateData.selected_items = [];
                if (itemSelect) itemSelect.value = "";
                const deployBtn = card.querySelector('[data-act=\"deploy\"]');
                if (deployBtn) deployBtn.disabled = true;
                send({
                    action: "toggle_items",
                    kind: uiState.activeKind,
                    gate: activeGate.key,
                    selected_items: [],
                });
            };
        }

        const testsWrap = card.querySelector('div[data-role="tests"]');
        testsWrap.innerHTML = '<div style="font-size:11px;color:var(--text-secondary);margin:8px 0 4px;">Tests</div>';
        if (!tests.length) {
            testsWrap.innerHTML += `<div class="dpw-empty">No tests registered for ${escapeHtml(kindLabel)}. Pass tests=... to deployment_pipeline_widget(), e.g. tests={"${escapeHtml(kind)}": {"My check": my_fn}} or tests={"all": {"My check": my_fn}}.</div>`;
        } else {
            tests.forEach(t => {
                const checked = selectedTests.has(t.name);
                const result = gateData.results[t.name];
                let statusHtml = "";
                if (result) {
                    if (result.error) statusHtml = '<span class="dpw-test-status error">ERROR</span>';
                    else if (result.ok) statusHtml = '<span class="dpw-test-status pass">PASS</span>';
                    else statusHtml = '<span class="dpw-test-status fail">FAIL</span>';
                }
                const row = document.createElement("div");
                row.innerHTML = `
                  <div class="dpw-test-row">
                    <input type="checkbox" data-test="${escapeHtml(t.name)}" ${checked ? "checked" : ""}/>
                    <code>${escapeHtml(t.name)}</code>
                    ${statusHtml}
                  </div>
                  ${result && result.error ? `<div class="dpw-test-err">${escapeHtml(result.error)}</div>` : ""}`;
                testsWrap.appendChild(row);
                row.querySelector("input").onchange = (e) => {
                    const cur = new Set(gateData.selected || []);
                    const testName = e.target.dataset.test;
                    if (e.target.checked) cur.add(testName);
                    else cur.delete(testName);
                    gateData.selected = Array.from(cur);
                    gateData.results = {};
                    const runBtn = card.querySelector('[data-act=\"run\"]');
                    if (runBtn) runBtn.disabled = gateData.selected.length === 0;
                    send({
                        action: "toggle_test",
                        kind: uiState.activeKind,
                        gate: activeGate.key,
                        selected: gateData.selected,
                    });
                };
            });
        }

        const selectedTestsList = gateData.selected || [];
        const hasSelectedTests = selectedTestsList.length > 0;
        const selectedItemCount = (gateData.selected_items || []).length;
        const hasOneItem = selectedItemCount === 1;
        const deployEnabled = hasOneItem;

        const actions = card.querySelector('div[data-role="actions"]');
        actions.innerHTML = `
          <button class="dpw-btn dpw-btn-ghost" data-act="run" ${hasSelectedTests ? "" : "disabled"}>Run Tests</button>
          <button class="dpw-btn dpw-btn-primary" data-act="deploy" ${deployEnabled ? "" : "disabled"}>Deploy</button>
        `;

        actions.querySelector('[data-act="run"]').onclick = () => send({
            action: "run_tests",
            kind: uiState.activeKind,
            gate: activeGate.key,
        });

        actions.querySelector('[data-act="deploy"]').onclick = () => {
            const itemSelect = card.querySelector('select[data-role="item-select"]');
            const selectedNow = itemSelect && itemSelect.value ? [itemSelect.value] : [];
            send({
                action: "deploy",
                kind: uiState.activeKind,
                gate: activeGate.key,
                source_stage_id: activeGate.source.id,
                target_stage_id: activeGate.target.id,
                selected_items: selectedNow,
            });
        };

        const n = data.notice || {};
        if (n.message) setStatus(n.message, n.kind || "info");
    }

    render();

    model.on("change:data", () => {
        setBusy(false);
        render();
    });

    model.on("change:status", () => {
        setBusy(false);
        const s = model.get("status") || {};
        setStatus(s.message, s.kind);
    });
}

export default { render };
"""

## 3. Widget class & data-loading helpers

In [ ]:
class _DeploymentPipelineWidget(anywidget.AnyWidget):
    _esm = _WIDGET_JS
    _css = _WIDGET_CSS

    data           = traitlets.Dict({}).tag(sync=True)
    status         = traitlets.Dict({}).tag(sync=True)
    pending_action = traitlets.Dict({}).tag(sync=True)
    run            = traitlets.Int(0).tag(sync=True)
    dark_mode      = traitlets.Bool(default_value=False, allow_none=True).tag(sync=True)

In [ ]:
# Compatibility shim: ensure deploy callable exists even if cells run out of order.
if "_labs_deploy" not in globals():
    import importlib
    from typing import Optional

    def _labs_deploy(**kwargs):
        last_err: Optional[Exception] = None
        candidates = [
            ("sempy_labs", "deploy_stage_content"),
            ("sempy_labs.deployment_pipeline", "deploy_stage_content"),
            ("sempy_labs.admin", "deploy_stage_content"),
        ]
        for mod_name, attr in candidates:
            try:
                mod = importlib.import_module(mod_name)
                fn = getattr(mod, attr)
                if callable(fn):
                    return fn(**kwargs)
            except Exception as exc:
                last_err = exc
                continue
        raise AttributeError(
            "Could not find sempy_labs function 'deploy_stage_content'. "
            f"Tried: {[m for m, _ in candidates]}. Last error: {last_err}"
        )

## 4. Public entry point `deployment_pipeline_widget(...)`

In [ ]:
import copy
import importlib
import inspect
import re
import sys
import traceback
from typing import Any, Callable, Dict, List, Optional


_ALL_KIND = "all"

# Map of human aliases -> canonical Fabric item-type string (or _ALL_KIND).
_KIND_ALIASES = {
    "all": _ALL_KIND,
    "both": _ALL_KIND,
    "any": _ALL_KIND,
    "*": _ALL_KIND,
    "report": "Report",
    "reports": "Report",
    "paginatedreport": "PaginatedReport",
    "paginated_report": "PaginatedReport",
    "dataset": "SemanticModel",
    "datasets": "SemanticModel",
    "semanticmodel": "SemanticModel",
    "semantic_model": "SemanticModel",
    "notebook": "Notebook",
    "lakehouse": "Lakehouse",
    "warehouse": "Warehouse",
    "datapipeline": "DataPipeline",
    "data_pipeline": "DataPipeline",
    "dataflow": "Dataflow",
    "dashboard": "Dashboard",
    "datamart": "Datamart",
    "kqldatabase": "KQLDatabase",
    "kql_database": "KQLDatabase",
    "kqlqueryset": "KQLQueryset",
    "eventstream": "Eventstream",
    "eventhouse": "Eventhouse",
    "mlmodel": "MLModel",
    "ml_model": "MLModel",
    "mlexperiment": "MLExperiment",
    "ml_experiment": "MLExperiment",
    "environment": "Environment",
    "sparkjobdefinition": "SparkJobDefinition",
    "spark_job_definition": "SparkJobDefinition",
    "mirroreddatabase": "MirroredDatabase",
    "mirrored_database": "MirroredDatabase",
    "reflex": "Reflex",
}

# Friendly display labels for known Fabric item types. Unknown types fall back
# to a humanised version of the raw type string.
_TYPE_LABELS = {
    "Report": "Report",
    "PaginatedReport": "Paginated report",
    "SemanticModel": "Semantic model",
    "Dataset": "Semantic model",
    "Dashboard": "Dashboard",
    "Dataflow": "Dataflow",
    "Datamart": "Datamart",
    "Notebook": "Notebook",
    "Lakehouse": "Lakehouse",
    "Warehouse": "Warehouse",
    "KQLDatabase": "KQL database",
    "KQLQueryset": "KQL queryset",
    "Eventstream": "Eventstream",
    "Eventhouse": "Eventhouse",
    "MLModel": "ML model",
    "MLExperiment": "ML experiment",
    "DataPipeline": "Data pipeline",
    "Environment": "Environment",
    "SparkJobDefinition": "Spark job definition",
    "MirroredDatabase": "Mirrored database",
    "Reflex": "Reflex (activator)",
}


def _normalize_kind(kind: Optional[str]) -> str:
    """Return canonical Fabric item-type string, _ALL_KIND, or the raw value."""
    if not kind:
        return _ALL_KIND
    key = str(kind).strip()
    lower = key.lower()
    if lower in _KIND_ALIASES:
        return _KIND_ALIASES[lower]
    return key


def _humanise_type(t: str) -> str:
    if not t:
        return ""
    if t in _TYPE_LABELS:
        return _TYPE_LABELS[t]
    parts = re.findall(r"[A-Z][a-z0-9]*|[a-z0-9]+", t)
    if not parts:
        return t
    return " ".join(p if i == 0 else p.lower() for i, p in enumerate(parts))


def dpw_test(kind: str = _ALL_KIND):
    """Mark a function as a deployment-pipeline widget test.

    Parameters
    ----------
    kind : str
        Either "all" / "both" (apply to every item type), or any Fabric item
        type string supported by deployment pipelines, e.g. "Report",
        "SemanticModel", "Notebook", "Lakehouse", "Warehouse", "DataPipeline",
        "KQLDatabase", "Dataflow", "PaginatedReport", "MirroredDatabase", ...

    Legacy aliases like "report" and "dataset" are accepted.

    A decorated test may accept zero args, or one positional arg which will
    receive the selected item dict: {"sourceItemId", "itemType", "itemName"}.
    The test should return a truthy value to pass.
    """
    normalized = _normalize_kind(kind)

    def deco(fn):
        fn.__dpw_kind__ = normalized
        return fn

    return deco


def _accepts_item(fn: Callable) -> bool:
    try:
        sig = inspect.signature(fn)
    except (TypeError, ValueError):
        return False
    positional = [
        p for p in sig.parameters.values()
        if p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)
    ]
    return len(positional) >= 1


def _discover_tests(explicit: Optional[Any]) -> Dict[str, Dict[str, Any]]:
    """Return tests map: {name: {"fn", "kind", "accepts_item"}}.

    Tests are *opt-in*. Accepted shapes for `explicit`:
      - {name: callable}                          (kind from @dpw_test, else "all")
      - {item_type: {name: callable}, ...}        (per-type nested dict; item_type
        may be a canonical Fabric type string, an alias, or "all"/"both")
    """
    if not explicit:
        return {}
    return _normalize_explicit_tests(explicit)


def _normalize_explicit_tests(explicit: Any) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    if not isinstance(explicit, dict) or not explicit:
        return out

    nested = all(isinstance(v, dict) for v in explicit.values())
    if nested:
        for kind, sub in explicit.items():
            normalized = _normalize_kind(kind)
            for name, fn in (sub or {}).items():
                if not callable(fn):
                    continue
                out[str(name)] = {
                    "fn": fn,
                    "kind": normalized,
                    "accepts_item": _accepts_item(fn),
                }
        return out

    for name, fn in explicit.items():
        if not callable(fn):
            continue
        kind = _normalize_kind(getattr(fn, "__dpw_kind__", _ALL_KIND))
        out[str(name)] = {
            "fn": fn,
            "kind": kind,
            "accepts_item": _accepts_item(fn),
        }
    return out


def _tests_state(
    tests_map: Dict[str, Dict[str, Any]],
    item_types: List[str],
) -> Dict[str, List[Dict[str, Any]]]:
    """Group tests by item type. _ALL_KIND tests appear on every type tab."""
    grouped: Dict[str, List[Dict[str, Any]]] = {t: [] for t in item_types}
    for name, meta in tests_map.items():
        entry = {"name": name, "accepts_item": bool(meta.get("accepts_item"))}
        kind = meta.get("kind", _ALL_KIND)
        if kind == _ALL_KIND:
            for t in item_types:
                grouped.setdefault(t, []).append(entry)
        else:
            grouped.setdefault(kind, []).append(entry)
    return grouped


def _recover_pandas_import_state() -> tuple[bool, str]:
    try:
        import pandas as pd
        _ = pd.DataFrame
        return True, ""
    except Exception as first_exc:
        sys.modules.pop("pandas", None)
        sys.modules.pop("pandas.core", None)
        importlib.invalidate_caches()
        try:
            import pandas as pd
            _ = pd.DataFrame
            return True, ""
        except Exception as second_exc:
            return False, (
                "Pandas import is still broken after a recovery attempt. "
                "Restart the kernel, then re-run from the install cell. "
                f"Initial error: {first_exc}. Retry error: {second_exc}"
            )


def _labs_fn(name: str):
    candidates = [
        ("sempy_labs", name),
        ("sempy_labs.deployment_pipeline", name),
        ("sempy_labs.admin", name),
    ]

    def _resolve_once():
        last_err: Optional[Exception] = None
        for mod_name, attr in candidates:
            try:
                mod = importlib.import_module(mod_name)
                fn = getattr(mod, attr)
                if callable(fn):
                    return fn, None
            except Exception as exc:
                last_err = exc
                continue
        return None, last_err

    fn, last_err = _resolve_once()
    if fn is not None:
        return fn

    err_text = str(last_err) if last_err is not None else ""
    if "partially initialized module 'pandas'" in err_text and "has no attribute 'core'" in err_text:
        ok, msg = _recover_pandas_import_state()
        if ok:
            fn, last_err = _resolve_once()
            if fn is not None:
                return fn
        if msg:
            raise RuntimeError(msg)

    raise AttributeError(
        f"Could not find sempy_labs function '{name}'. "
        f"Tried: {[m for m, _ in candidates]}. Last error: {last_err}"
    )


def _load_pipelines() -> List[Dict[str, str]]:
    fn = _labs_fn("list_deployment_pipelines")
    df = fn()
    if df is None or df.empty:
        return []
    cols = {c.lower(): c for c in df.columns}
    id_col = cols.get("deployment pipeline id") or list(df.columns)[0]
    name_col = cols.get("deployment pipeline name") or list(df.columns)[1]
    return [{"id": str(r[id_col]), "name": str(r[name_col])} for _, r in df.iterrows()]


def _load_stages(pipeline_id: str) -> List[Dict[str, str]]:
    fn = _labs_fn("list_deployment_pipeline_stages")
    df = fn(deployment_pipeline=pipeline_id)
    if df is None or df.empty:
        return []
    cols = {c.lower(): c for c in df.columns}
    id_col = cols.get("deployment pipeline stage id") or list(df.columns)[0]
    name_col = cols.get("deployment pipeline stage name") or list(df.columns)[1]
    ws_name = cols.get("workspace name")
    order = cols.get("order")
    rows = df.sort_values(order) if order else df
    return [
        {
            "id": str(r[id_col]),
            "name": str(r[name_col]),
            "workspace_name": (str(r[ws_name]) if ws_name and r[ws_name] is not None else ""),
        }
        for _, r in rows.iterrows()
    ]


def _load_stage_items_df(pipeline_id: str, stage_id: str):
    try:
        fn = _labs_fn("list_deployment_pipeline_stage_items")
        df = fn(deployment_pipeline=pipeline_id, stage=stage_id)
    except Exception:
        return None
    if df is None or df.empty:
        return None
    return df


def _load_stage_items_by_type(pipeline_id: str, stage_id: str) -> Dict[str, List[Dict[str, str]]]:
    """Return {item_type: [{sourceItemId, itemType, itemName}, ...]} for a stage.

    Works for *any* item type returned by the Fabric API, not just Report and
    SemanticModel.
    """
    df = _load_stage_items_df(pipeline_id, stage_id)
    if df is None:
        return {}
    cols = {c.lower(): c for c in df.columns}
    type_col = cols.get("item type")
    name_col = (
        cols.get("deployment pipeline stage item name")
        or cols.get("item display name")
        or cols.get("display name")
        or cols.get("name")
    )
    id_col_candidates = [
        cols.get("source item id"),
        cols.get("item id"),
        cols.get("deployment pipeline stage item id"),
    ]
    id_col_candidates = [c for c in id_col_candidates if c]
    if not type_col or not name_col or not id_col_candidates:
        return {}

    import pandas as _pd

    def _resolve_id(row) -> str:
        for c in id_col_candidates:
            v = row[c]
            if v is None:
                continue
            try:
                if _pd.isna(v):
                    continue
            except Exception:
                pass
            s = str(v).strip()
            if s and s.lower() not in ("none", "nan"):
                return s
        return ""

    out: Dict[str, List[Dict[str, str]]] = {}
    for _, r in df.iterrows():
        raw_type = r[type_col]
        try:
            if _pd.isna(raw_type):
                continue
        except Exception:
            pass
        itype = str(raw_type).strip()
        if not itype or itype.lower() in ("none", "nan"):
            continue
        sid = _resolve_id(r)
        if not sid:
            continue
        out.setdefault(itype, []).append({
            "sourceItemId": sid,
            "itemType": itype,
            "itemName": str(r[name_col]),
        })
    return out


def _labs_deploy(**kwargs):
    fn = _labs_fn("deploy_stage_content")
    return fn(**kwargs)


_FABRIC_ERROR_HINTS = {
    "Alm_InvalidRequest_MissingTargetReferencedDataset": (
        "The target stage is missing the semantic model that this item depends on. "
        "Deploy the underlying semantic model (dataset) to the target stage first, then retry."
    ),
    "Alm_InvalidRequest_MissingTargetReferencedReport": (
        "The target stage is missing a report that this item depends on. "
        "Deploy the dependent report to the target stage first."
    ),
    "Alm_InvalidRequest_MissingTargetReferencedDataflow": (
        "The target stage is missing a dataflow that this item depends on. "
        "Deploy the dependent dataflow to the target stage first."
    ),
    "Alm_InvalidRequest_MissingCapacityAssignment": (
        "The target stage's workspace is not assigned to a Fabric/Premium capacity. "
        "Assign a capacity to the target workspace and retry."
    ),
    "PrincipalDoesNotHavePipelineWritePermission": (
        "You don't have permission to deploy in this pipeline. Ask an admin to grant you pipeline write access."
    ),
}


def _extract_fabric_error_json(raw: str):
    import json as _json

    marker = raw.find("Error:")
    start = raw.find("{", marker if marker != -1 else 0)
    if start == -1:
        return None
    depth = 0
    in_str = False
    escape = False
    for i in range(start, len(raw)):
        ch = raw[i]
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return _json.loads(raw[start : i + 1])
                except Exception:
                    return None
    return None


def _format_deploy_error(exc: Exception, item: Dict[str, str], kind_label: str) -> str:
    item_name = item.get("itemName") or item.get("sourceItemId") or "(unknown)"
    raw = str(exc)
    code = None
    detail_code = None
    related = []
    server_message = None

    body = _extract_fabric_error_json(raw)
    if isinstance(body, dict):
        code = body.get("errorCode")
        server_message = body.get("message")
        for d in (body.get("moreDetails") or []):
            detail_code = detail_code or d.get("errorCode")
            rr = d.get("relatedResource") or {}
            if rr:
                related.append(f"{rr.get('resourceType','?')}/{rr.get('resourceId','?')}")

    hint = _FABRIC_ERROR_HINTS.get(detail_code or "") or _FABRIC_ERROR_HINTS.get(code or "")

    lines = [f"Deploy failed for {kind_label.lower()} '{item_name}'."]
    if hint:
        lines.append(hint)
    if code or detail_code:
        lines.append(f"Fabric error code: {detail_code or code}")
    if server_message and not hint:
        lines.append(f"Server message: {server_message}")
    if related:
        lines.append(f"Related resource(s): {', '.join(related)}")
    if not hint and not code:
        lines.append(traceback.format_exc(limit=8))
    return "\n".join(lines)


def _mk_gate_key(source_stage_id: str, target_stage_id: str) -> str:
    return f"{source_stage_id}__{target_stage_id}"


def _empty_gates_for_stages(stages: List[Dict[str, str]]) -> Dict[str, Dict[str, Any]]:
    gates: Dict[str, Dict[str, Any]] = {}
    for i in range(len(stages) - 1):
        key = _mk_gate_key(stages[i]["id"], stages[i + 1]["id"])
        gates[key] = {"selected": [], "selected_items": [], "results": {}}
    return gates


def _refresh_pipeline_state(state: Dict[str, Any], tests_map: Dict[str, Dict[str, Any]]) -> None:
    """(Re)load stages, items, item-types, gates and tests for the selected pipeline."""
    pid = state.get("selected_pipeline") or ""
    if not pid:
        state["stages"] = []
        state["items"] = {}
        state["item_types"] = []
        state["gates"] = {}
        state["tests"] = {}
        return

    stages = _load_stages(pid)
    state["stages"] = stages

    items_by_stage: Dict[str, Dict[str, List[Dict[str, str]]]] = {}
    type_counts: Dict[str, int] = {}
    for s in stages:
        by_type = _load_stage_items_by_type(pid, s["id"])
        items_by_stage[s["id"]] = by_type
        for t, lst in by_type.items():
            type_counts[t] = type_counts.get(t, 0) + len(lst)
    state["items"] = items_by_stage

    types_sorted = sorted(type_counts.keys(), key=lambda t: (-type_counts[t], t))
    state["item_types"] = [
        {"value": t, "label": _humanise_type(t), "count": type_counts[t]}
        for t in types_sorted
    ]

    empty_gates = _empty_gates_for_stages(stages)
    state["gates"] = {t: copy.deepcopy(empty_gates) for t in types_sorted}

    state["tests"] = _tests_state(tests_map, types_sorted)


def _build_widget_state(tests_map: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    notice: Dict[str, str] = {}
    try:
        pipelines = _load_pipelines()
    except Exception as exc:
        pipelines = []
        notice = {
            "kind": "error",
            "message": (
                "Could not load deployment pipelines. If this mentions pandas/core, "
                "restart the kernel and re-run from the install cell. "
                f"Details: {exc}"
            ),
        }

    default_id = pipelines[0]["id"] if pipelines else ""
    state: Dict[str, Any] = {
        "pipelines": pipelines,
        "selected_pipeline": default_id,
        "stages": [],
        "items": {},
        "item_types": [],
        "gates": {},
        "tests": {},
        "notice": notice,
    }
    if default_id:
        _refresh_pipeline_state(state, tests_map)
    return state


def _stage_items_for_type(data: Dict[str, Any], kind: str, source_stage_id: str) -> List[Dict[str, str]]:
    by_stage = data.get("items") or {}
    by_type = by_stage.get(source_stage_id) or {}
    return list(by_type.get(kind) or [])


def _selected_items_for_gate(
    data: Dict[str, Any], kind: str, gate: str, source_stage_id: str
) -> List[Dict[str, str]]:
    stage_items = _stage_items_for_type(data, kind, source_stage_id)
    gate_data = (data.get("gates", {}).get(kind, {}) or {}).get(gate, {})
    selected_ids = set(gate_data.get("selected_items") or [])
    seen: set[str] = set()
    out: List[Dict[str, str]] = []
    for x in stage_items:
        sid = str(x.get("sourceItemId") or "")
        if not sid or sid not in selected_ids or sid in seen:
            continue
        seen.add(sid)
        out.append(x)
    return out


def _resolve_selected_item(
    data: Dict[str, Any], kind: str, gate: str, source_stage_id: str
) -> Optional[Dict[str, str]]:
    items = _selected_items_for_gate(data, kind, gate, source_stage_id)
    if len(items) != 1:
        return None
    return items[0]


def _run_selected_tests(
    tests_map: Dict[str, Dict[str, Any]],
    selected: List[str],
    item: Optional[Dict[str, str]],
) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    for name in selected:
        meta = tests_map.get(name)
        if meta is None:
            out[name] = {"ok": False, "error": f"Test '{name}' not found."}
            continue
        fn = meta["fn"]
        accepts_item = bool(meta.get("accepts_item"))
        try:
            result = fn(item) if accepts_item else fn()
            out[name] = {"ok": bool(result)}
        except Exception:
            out[name] = {"ok": False, "error": traceback.format_exc(limit=6)}
    return out


_DEPLOYMENT_PIPELINE_WIDGET_INSTANCE = None


def deployment_pipeline_widget(
    tests: Optional[Any] = None,
    dark_mode: Optional[bool] = False,
):
    """Render the deployment pipeline widget for any Fabric item type a
    pipeline supports (Reports, Semantic models, Notebooks, Lakehouses,
    Warehouses, Data pipelines, KQL databases, Dataflows, etc.).

    Parameters
    ----------
    tests : dict, optional
        Either {name: callable} (kind from @dpw_test decorator, default "all")
        or {item_type: {name: callable}} where item_type is a Fabric item type
        string ("Report", "SemanticModel", "Notebook", "Lakehouse", ...) or
        "all" / "both". Legacy aliases "report" and "dataset" map to "Report"
        and "SemanticModel".
    dark_mode : bool or None
        True for dark, False for light, None to follow the host theme.
    """
    from IPython.display import display

    tests_map = _discover_tests(tests)
    state = _build_widget_state(tests_map)

    global _DEPLOYMENT_PIPELINE_WIDGET_INSTANCE
    widget = _DEPLOYMENT_PIPELINE_WIDGET_INSTANCE
    created_now = False
    if widget is None:
        widget = _DeploymentPipelineWidget()
        _DEPLOYMENT_PIPELINE_WIDGET_INSTANCE = widget
        created_now = True

    def _publish(new_data: Dict[str, Any]) -> None:
        """Assign widget.data with a deep copy so traitlets detects the change."""
        widget.data = copy.deepcopy(new_data)

    widget.dark_mode = dark_mode
    _publish(state)
    widget.status = {"kind": "info", "message": "Widget ready."}

    def _set_status(kind: str, message: str) -> None:
        widget.status = {"kind": kind, "message": message}

    def _kind_label(data: Dict[str, Any], kind: str) -> str:
        for t in data.get("item_types") or []:
            if t.get("value") == kind:
                return t.get("label") or kind
        return _humanise_type(kind) or kind

    def _handle_action(action: Dict[str, Any]) -> None:
        if not action:
            return

        act = action.get("action")
        kind = action.get("kind")
        data = copy.deepcopy(widget.data)

        if act == "refresh_pipelines":
            new_state = _build_widget_state(tests_map)
            _publish(new_state)
            _set_status("success", "Pipelines refreshed.")
            return

        if act == "select_pipeline":
            pid = action.get("pipeline_id") or ""
            data["selected_pipeline"] = pid
            _refresh_pipeline_state(data, tests_map)
            _publish(data)
            if pid:
                _set_status("info", "Pipeline updated.")
            else:
                _set_status("info", "Pipeline cleared.")
            return

        if act == "toggle_items" and kind:
            gate = action.get("gate") or ""
            raw_items = action.get("selected_items") or []
            if isinstance(raw_items, str):
                selected_items = [raw_items]
            else:
                selected_items = list(raw_items)
            selected_items = [str(x) for x in selected_items if x]
            if selected_items:
                selected_items = [selected_items[0]]
            gates = data.get("gates", {}).get(kind, {})
            if gate in gates:
                gates[gate]["selected_items"] = selected_items
                # Item changed -> previous test results no longer apply.
                gates[gate]["results"] = {}
                _publish(data)
            return

        if act == "toggle_test" and kind:
            gate = action.get("gate") or ""
            selected = list(action.get("selected") or [])
            gates = data.get("gates", {}).get(kind, {})
            if gate in gates:
                gates[gate]["selected"] = selected
                gates[gate]["results"] = {}
                _publish(data)
            return

        if act == "run_tests" and kind:
            gate = action.get("gate") or ""
            gates = data.get("gates", {}).get(kind, {})
            if gate not in gates:
                _set_status("error", "Gate not found.")
                return
            selected = list(gates[gate].get("selected") or [])
            if not selected:
                _set_status("error", "Select at least one test first.")
                return

            source_stage_id = gate.split("__", 1)[0] if "__" in gate else ""
            item = _resolve_selected_item(data, kind, gate, source_stage_id)
            needs_item = any(
                (tests_map.get(n) or {}).get("accepts_item") for n in selected
            )
            if needs_item and item is None:
                _set_status("error", "Select exactly one item before running tests.")
                return

            gates[gate]["results"] = _run_selected_tests(tests_map, selected, item)
            _publish(data)
            failed = [n for n in selected if not gates[gate]["results"].get(n, {}).get("ok", False)]
            if not failed:
                _set_status("success", "All selected tests passed.")
            else:
                _set_status("error", f"Failed tests: {', '.join(failed)}")
            return

        if act == "deploy" and kind:
            gate = action.get("gate") or ""
            source = action.get("source_stage_id")
            target = action.get("target_stage_id")

            gates = data.get("gates", {}).get(kind, {})
            gate_data = gates.get(gate)
            if not gate_data:
                _set_status("error", "Gate not found.")
                return

            selected_tests = list(gate_data.get("selected") or [])
            if selected_tests:
                source_stage_id = gate.split("__", 1)[0] if "__" in gate else ""
                item_for_tests = _resolve_selected_item(data, kind, gate, source_stage_id)
                needs_item = any(
                    (tests_map.get(n) or {}).get("accepts_item") for n in selected_tests
                )
                if needs_item and item_for_tests is None:
                    _set_status("error", "Select exactly one item before deploy.")
                    return

                existing_results = gate_data.get("results") or {}
                missing = [n for n in selected_tests if n not in existing_results]
                if missing:
                    new_results = dict(existing_results)
                    new_results.update(_run_selected_tests(tests_map, missing, item_for_tests))
                    gate_data["results"] = new_results
                    _publish(data)

                results = gate_data.get("results") or {}
                failed_tests = [n for n in selected_tests if not results.get(n, {}).get("ok", False)]
                if failed_tests:
                    _set_status("error", f"Tests failed, deploy aborted: {', '.join(failed_tests)}")
                    return

            pid = data.get("selected_pipeline") or ""
            if not pid or not source or not target:
                _set_status("error", "Missing pipeline or stage selection.")
                return

            raw_items = action.get("selected_items") or []
            if isinstance(raw_items, str):
                override = [raw_items]
            else:
                override = list(raw_items)
            override = [str(x) for x in override if x]
            if override:
                override = [override[0]]
            gate_data["selected_items"] = override
            items = _selected_items_for_gate(data, kind, gate, source)
            if len(items) != 1:
                if len(items) == 0:
                    _set_status("error", "Select exactly one item to deploy.")
                else:
                    _set_status("error", "Select only one item to deploy.")
                return

            payload = [{
                "sourceItemId": items[0]["sourceItemId"],
                "itemType": items[0]["itemType"],
            }]
            label = _kind_label(data, kind)

            try:
                _labs_deploy(
                    deployment_pipeline=pid,
                    source_stage_id=str(source),
                    target_stage_id=str(target),
                    items=payload,
                    note=f"Promoted via deployment_pipeline_widget ({label})",
                )
            except Exception as exc:
                msg = _format_deploy_error(exc, items[0], label)
                _set_status("error", msg)
                return

            _set_status("success", f"Deployed 1 {label.lower()} item.")
            return

    def _on_run(change: Dict[str, Any]) -> None:
        try:
            _handle_action(widget.pending_action or {})
        except Exception:
            _set_status("error", traceback.format_exc(limit=8))

    # Always (re-)install the action handler. When the widget instance is cached
    # across calls, the previous closure captured the *previous* tests_map, so
    # newly-passed tests would be unrecognized ("Test not found"). Swapping the
    # observer rebinds the handler to the current tests_map.
    prev_handler = getattr(widget, "_dpw_on_run", None)
    if prev_handler is not None:
        try:
            widget.unobserve(prev_handler, names="run")
        except Exception:
            pass
    widget.observe(_on_run, names="run")
    widget._dpw_on_run = _on_run

    if created_now:
        display(widget)
        # Return None so a direct function call does not trigger a second rich display.
        return None

    return widget

## 5. Define your tests

Tests are *opt-in*. Decorate functions with `@dpw_test(kind=...)` and pass them explicitly to `deployment_pipeline_widget(tests=...)` (see the example at the bottom).

`kind` can be:
- `"all"` (or `"both"`) — show the test on every item-type tab.
- Any Fabric item type string — e.g. `"Report"`, `"SemanticModel"`, `"Notebook"`, `"Lakehouse"`, `"Warehouse"`, `"DataPipeline"`, `"KQLDatabase"`, `"Dataflow"`, `"PaginatedReport"`, `"MirroredDatabase"`, ...
- Legacy aliases `"report"` and `"dataset"` still work.

Each test may take no args, or one positional arg that receives the selected item dict (`{"sourceItemId", "itemType", "itemName"}`).

In [ ]:
@dpw_test(kind="SemanticModel")
def model_has_measures(item):
    """SemanticModel-scoped test. Replace with real model validation for the given item."""
    return True

@dpw_test(kind="all")
def best_practice_analyzer_clean(item):
    """Applies to every item type. Replace with: run BPA on the item and check no critical violations."""
    return True

@dpw_test(kind="Report")
def report_pages_render(item):
    """Report-scoped test. Replace with: render-check across report pages for the given item."""
    return True

@dpw_test(kind="Report")
def no_broken_visuals(item):
    """Report-scoped test. Replace with: scan visuals for errors on the given item."""
    return True

@dpw_test(kind="Notebook")
def notebook_compiles(item):
    """Notebook-scoped test. Replace with: parse-check the notebook definition for the given item."""
    return True

@dpw_test(kind="Lakehouse")
def lakehouse_has_tables(item):
    """Lakehouse-scoped test. Replace with: assert the lakehouse has the expected tables."""
    return True

@dpw_test(kind="DataPipeline")
def pipeline_activities_valid(item):
    """DataPipeline-scoped test. Replace with: validate pipeline activities for the given item."""
    return True

## 6. Launch the widget

### Passing tests explicitly (optional)

In [ ]:
# Per-item-type dict. Keys can be any Fabric item type string the pipeline
# supports — e.g. Report, SemanticModel, Notebook, Lakehouse, Warehouse,
# DataPipeline, KQLDatabase, Dataflow, PaginatedReport, MirroredDatabase, ...
# Use "all" (or "both") to apply a test to every type tab.
custom_tests = {
    "Report": {
        "Report pages render": report_pages_render,
        "No broken visuals": no_broken_visuals,
    },
    "SemanticModel": {
        "Model has measures": model_has_measures,
    },
    "Notebook": {
        "Notebook compiles": notebook_compiles,
    },
    "Lakehouse": {
        "Lakehouse has tables": lakehouse_has_tables,
    },
    "DataPipeline": {
        "Pipeline activities valid": pipeline_activities_valid,
    },
    "all": {
        "BPA clean": best_practice_analyzer_clean,
    },
}

# To launch with this explicit map, run:
deployment_pipeline_widget(tests=custom_tests, dark_mode=None)